# Assignment 3 – ACID Transactions on a B+ Tree Database

This notebook demonstrates full ACID transaction support built on top of the existing B+ Tree storage engine. It utilizes a highly relational Sports Management schema with strict constraints and Foreign Key relationships.

### Tables used:
| Table | Primary Key | Constraints & Relationships |
|-------|-------------|-----------------------------|
| `member` | `MemberID` | CHECK ('x > 0'), UNIQUE Email |
| `sport` | `SportID` | UNIQUE SportName, CHECK ('x is None or x > 0') |
| `team` | `TeamID` | FK to Member (Coach, Captain), FK to Sport |
| `equipment`| `EquipmentID` | CHECK ('x >= 0'), FK to Sport |

## 0. Setup – Create database, tables, and seed data

**Note:** Constraints are now defined as string expressions (e.g., `'x > 0'`) to allow for native Python pickling and complete durability upon restarts.

In [114]:
%load_ext autoreload
%autoreload 2

from db_manager import DatabaseManager
from transaction_manager import TransactionManager, ConstraintError, TransactionError
import time
import threading

# ── bootstrap ─────────────────────────────────────────────────────────
db = DatabaseManager()
db.create_database('sports_db')

# 1. Create Tables
db.create_table('sports_db', 'member', {
    'MemberID': int, 'Name': str, 'Image': str, 'Age': int, 'Email': str, 
    'ContactNumber': str, 'Gender': str, 'Role': str, 'JoinDate': str
}, order=8, search_key='MemberID', constraints={'Age': {'CHECK': ['x > 0']}, 'Email': {'UNIQUE': True, 'NOT NULL': True}}, referenced_by=[('team', 'CoachID'), ('team', 'CaptainID')])

db.create_table('sports_db', 'sport', {
    'SportID': int, 'SportName': str, 'Category': str, 'MaxPlayersPerTeam': (int, type(None))
}, order=8, search_key='SportID', constraints={'MaxPlayersPerTeam': {'CHECK': ['x is None or x > 0']}}, referenced_by=[('team', 'SportID'), ('equipment', 'SportID')])

db.create_table('sports_db', 'team', {
    'TeamID': int, 'TeamName': str, 'CoachID': (int, type(None)), 'CaptainID': (int, type(None)),
    'SportID': int, 'FormedDate': str
}, order=8, search_key='TeamID', foreign_keys={'CoachID': 'member', 'CaptainID': 'member', 'SportID': 'sport'})

db.create_table('sports_db', 'equipment', {
    'EquipmentID': int, 'EquipmentName': str, 'TotalQuantity': int, 
    'EquipmentCondition': str, 'SportID': (int, type(None))
}, order=8, search_key='EquipmentID', constraints={'TotalQuantity': {'CHECK': ['x >= 0'], 'NOT NULL': True}}, foreign_keys={'SportID': 'sport'})

# 2. Fetch table handles for seeding and transaction demos
member_tbl, _ = db.get_table('sports_db', 'member')
sport_tbl, _ = db.get_table('sports_db', 'sport')
team_tbl, _ = db.get_table('sports_db', 'team')
eq_tbl, _ = db.get_table('sports_db', 'equipment')

# FIX: Constraints converted to eval-safe strings to prevent lambda pickle crashes

# ── seed data ─────────────────────────────────────────────────────────
members_data = [
    {'MemberID': 1, 'Name': 'Aarav Sharma', 'Image': 'img/aarav.jpg', 'Age': 19, 'Email': 'aarav.sharma@iitgn.ac.in', 'ContactNumber': '9876543201', 'Gender': 'M', 'Role': 'Player', 'JoinDate': '2024-08-01'},
    {'MemberID': 2, 'Name': 'Meera Patel', 'Image': 'img/meera.jpg', 'Age': 20, 'Email': 'meera.patel@iitgn.ac.in', 'ContactNumber': '9876543202', 'Gender': 'F', 'Role': 'Player', 'JoinDate': '2024-08-01'},
    {'MemberID': 3, 'Name': 'Rohan Das', 'Image': 'img/rohan.jpg', 'Age': 21, 'Email': 'rohan.das@iitgn.ac.in', 'ContactNumber': '9876543203', 'Gender': 'M', 'Role': 'Player', 'JoinDate': '2024-08-15'},
    {'MemberID': 4, 'Name': 'Priya Singh', 'Image': 'img/priya.jpg', 'Age': 19, 'Email': 'priya.singh@iitgn.ac.in', 'ContactNumber': '9876543204', 'Gender': 'F', 'Role': 'Player', 'JoinDate': '2024-09-01'},
    {'MemberID': 5, 'Name': 'Arjun Mehta', 'Image': 'img/arjun.jpg', 'Age': 22, 'Email': 'arjun.mehta@iitgn.ac.in', 'ContactNumber': '9876543205', 'Gender': 'M', 'Role': 'Player', 'JoinDate': '2024-08-10'},
    {'MemberID': 6, 'Name': 'Kavya Iyer', 'Image': 'img/kavya.jpg', 'Age': 20, 'Email': 'kavya.iyer@iitgn.ac.in', 'ContactNumber': '9876543206', 'Gender': 'F', 'Role': 'Player', 'JoinDate': '2024-08-20'},
    {'MemberID': 7, 'Name': 'Vikash Yadav', 'Image': 'img/vikash.jpg', 'Age': 21, 'Email': 'vikash.yadav@iitgn.ac.in', 'ContactNumber': '9876543207', 'Gender': 'M', 'Role': 'Player', 'JoinDate': '2024-09-05'},
    {'MemberID': 8, 'Name': 'Ananya Joshi', 'Image': 'img/ananya.jpg', 'Age': 19, 'Email': 'ananya.joshi@iitgn.ac.in', 'ContactNumber': '9876543208', 'Gender': 'F', 'Role': 'Player', 'JoinDate': '2024-09-10'},
    {'MemberID': 9, 'Name': 'Siddharth Nair', 'Image': 'img/siddharth.jpg', 'Age': 23, 'Email': 'siddharth.nair@iitgn.ac.in', 'ContactNumber': '9876543209', 'Gender': 'M', 'Role': 'Player', 'JoinDate': '2024-08-05'},
    {'MemberID': 10, 'Name': 'Riya Gupta', 'Image': 'img/riya.jpg', 'Age': 20, 'Email': 'riya.gupta@iitgn.ac.in', 'ContactNumber': '9876543210', 'Gender': 'F', 'Role': 'Player', 'JoinDate': '2024-08-25'},
    {'MemberID': 11, 'Name': 'Harsh Pandey', 'Image': 'img/harsh.jpg', 'Age': 22, 'Email': 'harsh.pandey@iitgn.ac.in', 'ContactNumber': '9876543211', 'Gender': 'M', 'Role': 'Player', 'JoinDate': '2024-09-15'},
    {'MemberID': 12, 'Name': 'Diya Kapoor', 'Image': 'img/diya.jpg', 'Age': 18, 'Email': 'diya.kapoor@iitgn.ac.in', 'ContactNumber': '9876543212', 'Gender': 'F', 'Role': 'Player', 'JoinDate': '2024-10-01'},
    {'MemberID': 13, 'Name': 'Raj Kumar', 'Image': 'img/raj.jpg', 'Age': 38, 'Email': 'raj.kumar@iitgn.ac.in', 'ContactNumber': '9876543213', 'Gender': 'M', 'Role': 'Coach', 'JoinDate': '2023-06-15'},
    {'MemberID': 14, 'Name': 'Sunita Nair', 'Image': 'img/sunita.jpg', 'Age': 35, 'Email': 'sunita.nair@iitgn.ac.in', 'ContactNumber': '9876543214', 'Gender': 'F', 'Role': 'Coach', 'JoinDate': '2023-07-01'},
    {'MemberID': 15, 'Name': 'Deepak Verma', 'Image': 'img/deepak.jpg', 'Age': 42, 'Email': 'deepak.verma@iitgn.ac.in', 'ContactNumber': '9876543215', 'Gender': 'M', 'Role': 'Coach', 'JoinDate': '2023-05-20'},
    {'MemberID': 16, 'Name': 'Pooja Rathi', 'Image': 'img/pooja.jpg', 'Age': 36, 'Email': 'pooja.rathi@iitgn.ac.in', 'ContactNumber': '9876543216', 'Gender': 'F', 'Role': 'Coach', 'JoinDate': '2023-08-10'},
    {'MemberID': 17, 'Name': 'Manoj Tiwari', 'Image': 'img/manoj.jpg', 'Age': 45, 'Email': 'manoj.tiwari@iitgn.ac.in', 'ContactNumber': '9876543217', 'Gender': 'M', 'Role': 'Coach', 'JoinDate': '2023-04-01'},
    {'MemberID': 18, 'Name': 'Vikram Reddy', 'Image': 'img/vikram.jpg', 'Age': 40, 'Email': 'vikram.reddy@iitgn.ac.in', 'ContactNumber': '9876543218', 'Gender': 'M', 'Role': 'Admin', 'JoinDate': '2023-01-10'},
    {'MemberID': 19, 'Name': 'Neha Agarwal', 'Image': 'img/neha.jpg', 'Age': 32, 'Email': 'neha.agarwal@iitgn.ac.in', 'ContactNumber': '9876543219', 'Gender': 'F', 'Role': 'Admin', 'JoinDate': '2023-02-15'},
    {'MemberID': 20, 'Name': 'Amit Choudhary', 'Image': 'img/amit.jpg', 'Age': 34, 'Email': 'amit.choudhary@iitgn.ac.in', 'ContactNumber': '9876543220', 'Gender': 'M', 'Role': 'Admin', 'JoinDate': '2023-03-01'}
]
for m in members_data: member_tbl.insert(m)

sports_data = [
    {'SportID': 1, 'SportName': 'Athletics', 'Category': 'Individual', 'MaxPlayersPerTeam': None},
    {'SportID': 2, 'SportName': 'Football', 'Category': 'Team', 'MaxPlayersPerTeam': 11},
    {'SportID': 3, 'SportName': 'Badminton', 'Category': 'Dual', 'MaxPlayersPerTeam': 2},
    {'SportID': 4, 'SportName': 'Cricket', 'Category': 'Team', 'MaxPlayersPerTeam': 11},
    {'SportID': 5, 'SportName': 'Basketball', 'Category': 'Team', 'MaxPlayersPerTeam': 5},
    {'SportID': 6, 'SportName': 'Tennis', 'Category': 'Dual', 'MaxPlayersPerTeam': 2},
    {'SportID': 7, 'SportName': 'Volleyball', 'Category': 'Team', 'MaxPlayersPerTeam': 6},
    {'SportID': 8, 'SportName': 'Swimming', 'Category': 'Individual', 'MaxPlayersPerTeam': None}
]
for s in sports_data: sport_tbl.insert(s)

teams_data = [
    {'TeamID': 1, 'TeamName': 'Thunder Sprinters (Aarav)', 'CoachID': 13, 'CaptainID': 1, 'SportID': 1, 'FormedDate': '2024-08-15'},
    {'TeamID': 2, 'TeamName': 'Thunder Sprinters (Arjun)', 'CoachID': 13, 'CaptainID': 5, 'SportID': 1, 'FormedDate': '2024-08-15'},
    {'TeamID': 3, 'TeamName': 'Thunder Sprinters (Sid)', 'CoachID': 13, 'CaptainID': 9, 'SportID': 1, 'FormedDate': '2024-08-15'},
    {'TeamID': 4, 'TeamName': 'Shuttle Stars (Meera)', 'CoachID': 14, 'CaptainID': 2, 'SportID': 3, 'FormedDate': '2024-08-20'},
    {'TeamID': 5, 'TeamName': 'Shuttle Stars (Kavya)', 'CoachID': 14, 'CaptainID': 6, 'SportID': 3, 'FormedDate': '2024-08-20'},
    {'TeamID': 6, 'TeamName': 'Tennis Aces (Priya)', 'CoachID': 16, 'CaptainID': 4, 'SportID': 6, 'FormedDate': '2024-09-05'},
    {'TeamID': 7, 'TeamName': 'Tennis Aces (Ananya)', 'CoachID': 16, 'CaptainID': 8, 'SportID': 6, 'FormedDate': '2024-09-05'},
    {'TeamID': 8, 'TeamName': 'Aqua Sharks (Diya)', 'CoachID': 14, 'CaptainID': 12, 'SportID': 8, 'FormedDate': '2024-09-15'},
    {'TeamID': 9, 'TeamName': 'IITGN FC Alpha', 'CoachID': 15, 'CaptainID': 7, 'SportID': 2, 'FormedDate': '2024-08-10'},
    {'TeamID': 10, 'TeamName': 'IITGN FC Beta', 'CoachID': 15, 'CaptainID': 3, 'SportID': 2, 'FormedDate': '2024-10-01'},
    {'TeamID': 11, 'TeamName': 'Cricket XI Lions', 'CoachID': 17, 'CaptainID': 11, 'SportID': 4, 'FormedDate': '2024-08-12'},
    {'TeamID': 12, 'TeamName': 'Hoop Warriors', 'CoachID': 16, 'CaptainID': 1, 'SportID': 5, 'FormedDate': '2024-09-01'},
    {'TeamID': 13, 'TeamName': 'Volley Vipers', 'CoachID': 13, 'CaptainID': 2, 'SportID': 7, 'FormedDate': '2024-09-10'},
    {'TeamID': 14, 'TeamName': 'Shuttle Queens (Doubles)', 'CoachID': 14, 'CaptainID': 2, 'SportID': 3, 'FormedDate': '2024-09-01'},
    {'TeamID': 15, 'TeamName': 'Tennis Duo (Doubles)', 'CoachID': 16, 'CaptainID': 4, 'SportID': 6, 'FormedDate': '2024-09-01'}
]
for t in teams_data: team_tbl.insert(t)

equipment_data = [
    {'EquipmentID': 1, 'EquipmentName': 'Football', 'TotalQuantity': 20, 'EquipmentCondition': 'Good', 'SportID': 2},
    {'EquipmentID': 2, 'EquipmentName': 'Badminton Racket', 'TotalQuantity': 15, 'EquipmentCondition': 'Good', 'SportID': 3},
    {'EquipmentID': 3, 'EquipmentName': 'Shuttlecock (Box)', 'TotalQuantity': 30, 'EquipmentCondition': 'New', 'SportID': 3},
    {'EquipmentID': 4, 'EquipmentName': 'Cricket Bat', 'TotalQuantity': 10, 'EquipmentCondition': 'Good', 'SportID': 4},
    {'EquipmentID': 5, 'EquipmentName': 'Cricket Ball (Box)', 'TotalQuantity': 25, 'EquipmentCondition': 'New', 'SportID': 4},
    {'EquipmentID': 6, 'EquipmentName': 'Basketball', 'TotalQuantity': 12, 'EquipmentCondition': 'Good', 'SportID': 5},
    {'EquipmentID': 7, 'EquipmentName': 'Tennis Racket', 'TotalQuantity': 8, 'EquipmentCondition': 'Fair', 'SportID': 6},
    {'EquipmentID': 8, 'EquipmentName': 'Tennis Ball (Can)', 'TotalQuantity': 20, 'EquipmentCondition': 'New', 'SportID': 6},
    {'EquipmentID': 9, 'EquipmentName': 'Volleyball', 'TotalQuantity': 10, 'EquipmentCondition': 'Good', 'SportID': 7},
    {'EquipmentID': 10, 'EquipmentName': 'Swimming Goggles', 'TotalQuantity': 15, 'EquipmentCondition': 'New', 'SportID': 8},
    {'EquipmentID': 11, 'EquipmentName': 'Starting Blocks', 'TotalQuantity': 6, 'EquipmentCondition': 'Good', 'SportID': 1},
    {'EquipmentID': 12, 'EquipmentName': 'Stopwatch', 'TotalQuantity': 10, 'EquipmentCondition': 'Good', 'SportID': None},
    {'EquipmentID': 13, 'EquipmentName': 'First Aid Kit', 'TotalQuantity': 5, 'EquipmentCondition': 'Good', 'SportID': None},
    {'EquipmentID': 14, 'EquipmentName': 'Cone Markers (Set)', 'TotalQuantity': 20, 'EquipmentCondition': 'New', 'SportID': None},
    {'EquipmentID': 15, 'EquipmentName': 'Resistance Bands', 'TotalQuantity': 12, 'EquipmentCondition': 'New', 'SportID': None}
]
for e in equipment_data: eq_tbl.insert(e)

db.save_to_disk('database.dat')
# Transaction Manager (creates WAL)
tm = TransactionManager(db, wal_path='wal.log')

def read_record(tm, db_name, table_name, record_id):
    tx = tm.begin()
    try:
        record = tm.read(tx, db_name, table_name, record_id)
        tm.commit(tx)
        return record
    except Exception:
        if tx.status.name == 'ACTIVE':
            tm.rollback(tx)
        raise

def record_exists(tm, db_name, table_name, record_id):
    return read_record(tm, db_name, table_name, record_id) is not None

def show(label, tbl, limit=5):
    print(f'\n── {label} (Showing {limit}) ─────────────────────────')
    for _, v in list(tbl.get_all())[:limit]:
        print(' ', v)

print('\nSetup complete. All 20 Members, 8 Sports, 15 Teams, and 15 Equipment items injected.')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

Setup complete. All 20 Members, 8 Sports, 15 Teams, and 15 Equipment items injected.


---
## A1. Atomicity – Multi-table COMMIT (Success)

A single transaction to setup a new Cricket initiative (Touches 3 relations):
1. Add 2 `Cricket Bats` from Equipment (ID 4)
2. Updates `Harsh Pandey` (Member 11) to be a 'Coach'
3. Inserts a new `Cricket XI Beta` Team (ID 16) pointing to Harsh.

All three succeed → COMMIT.

In [115]:
tx = tm.begin()
try:
    print("Starting multi-table transaction to add Cricket Bats, promote Harsh, and create new Team...")
    # Step 1: Add Cricket Bats (Relation 1)
    bat = tm.read(tx, 'sports_db', 'equipment', 4)
    tm.update(tx, 'sports_db', 'equipment', 4, {**bat, 'TotalQuantity': bat['TotalQuantity'] + 2})
    
    # Step 2: Promote Harsh Pandey to Coach (Relation 2)
    harsh = tm.read(tx, 'sports_db', 'member', 11)
    tm.update(tx, 'sports_db', 'member', 11, {**harsh, 'Role': 'Coach'})
    
    # Step 3: Create new Team (Relation 3)
    tm.insert(tx, 'sports_db', 'team', 16, {
        'TeamID': 16, 'TeamName': 'Cricket XI Beta', 'CoachID': 11, 
        'CaptainID': 11, 'SportID': 4, 'FormedDate': '2024-11-01'
    })
    
    tm.commit(tx)
    print("Success! Multi-table transaction committed.\n")
except Exception as e:
    print(f"Transaction failed: {e}")

print("Reading records...")
print("\nCricket Bat Quantity:", read_record(tm, 'sports_db', 'equipment', 4)['TotalQuantity'])
print("---------------")
print("\nHarsh's Role:", read_record(tm, 'sports_db', 'member', 11)['Role'])
print("---------------")
print("\nNew Team exists:", record_exists(tm, 'sports_db', 'team', 16))
print("---------------")

[TxManager] BEGIN  tx=621b78dc
Starting multi-table transaction to add Cricket Bats, promote Harsh, and create new Team...
[TxManager] COMMIT tx=621b78dc
Success! Multi-table transaction committed.

Reading records...
[TxManager] BEGIN  tx=d94c82ad
[TxManager] COMMIT tx=d94c82ad

Cricket Bat Quantity: 12
---------------
[TxManager] BEGIN  tx=bd4938ac
[TxManager] COMMIT tx=bd4938ac

Harsh's Role: Coach
---------------
[TxManager] BEGIN  tx=4ee31fbd
[TxManager] COMMIT tx=4ee31fbd

New Team exists: True
---------------


---
## A2. Atomicity – Multi-table Failure → Automatic ROLLBACK

We attempt a multi-table transaction (Touches 3 relations) that **fails midway**. 
1. Deduct 5 Basketballs (Valid)
2. Promote `Aarav Sharma` to Coach (Valid)
3. Insert a new Team, but assign a `CoachID` (9999) that does NOT exist in the Members table!

The `TransactionManager` will catch the Foreign Key constraint violation on step 3, abort the transaction, and safely roll back the first two tables to their original state.

In [116]:
print('=== State BEFORE failing transaction ===')
print('Read transaction to confirm initial state before attempting failing transaction...')
print('Basketball Quantity:', read_record(tm, 'sports_db', 'equipment', 6)['TotalQuantity']) # Should be 12
print('---------------')
print('Aarav Role:', read_record(tm, 'sports_db', 'member', 1)['Role']) # Should be Player
print('---------------')

tx2 = tm.begin()
try:
    print("\n[Tx] 1. Deducting 5 Basketballs...")
    bb = tm.read(tx2, 'sports_db', 'equipment', 6)
    tm.update(tx2, 'sports_db', 'equipment', 6, {**bb, 'TotalQuantity': bb['TotalQuantity'] - 5})

    print("[Tx] 2. Promoting Aarav to Coach...")
    aarav = tm.read(tx2, 'sports_db', 'member', 1)
    tm.update(tx2, 'sports_db', 'member', 1, {**aarav, 'Role': 'Coach'})

    print("[Tx] 3. Attempting to insert Team with ID 17 and Invalid CoachID (9999)...")
    tm.insert(tx2, 'sports_db', 'team', 17, {
        'TeamID': 17, 'TeamName': 'Hoop Legends', 'CoachID': 9999, # INVALID!
        'CaptainID': 1, 'SportID': 5, 'FormedDate': '2024-11-05'
    })
    
    tm.commit(tx2)
except ConstraintError as e:
    print(f"\n*** ERROR CAUGHT: {e} ***")
    print("*** TransactionManager automatically triggered ROLLBACK! ***")

print('\n=== State AFTER rollback (must match BEFORE) ===')
print('Basketball Quantity:', read_record(tm, 'sports_db', 'equipment', 6)['TotalQuantity']) # Must be 12 again
print('---------------')
print('Aarav Role:', read_record(tm, 'sports_db', 'member', 1)['Role']) # Must be Player again
print('---------------')
print('Team with ID 17 exists:', record_exists(tm, 'sports_db', 'team', 17)) # Must be False

=== State BEFORE failing transaction ===
Read transaction to confirm initial state before attempting failing transaction...
[TxManager] BEGIN  tx=bfc04a28
[TxManager] COMMIT tx=bfc04a28
Basketball Quantity: 12
---------------
[TxManager] BEGIN  tx=7ff8368d
[TxManager] COMMIT tx=7ff8368d
Aarav Role: Player
---------------
[TxManager] BEGIN  tx=727a412b

[Tx] 1. Deducting 5 Basketballs...
[Tx] 2. Promoting Aarav to Coach...
[Tx] 3. Attempting to insert Team with ID 17 and Invalid CoachID (9999)...
[TxManager] ABORT  tx=727a412b

*** ERROR CAUGHT: Foreign Key Violation: 'CoachID'=9999 does not exist in 'member' ***
*** TransactionManager automatically triggered ROLLBACK! ***

=== State AFTER rollback (must match BEFORE) ===
[TxManager] BEGIN  tx=2b6d02e1
[TxManager] COMMIT tx=2b6d02e1
Basketball Quantity: 12
---------------
[TxManager] BEGIN  tx=8ad30cf4
[TxManager] COMMIT tx=8ad30cf4
Aarav Role: Player
---------------
[TxManager] BEGIN  tx=bc6d31ee
[TxManager] COMMIT tx=bc6d31ee
Team wit

---
## A3. Atomicity – Primary Key Shift Rollback

If a transaction updates a record's Primary Key (e.g., shifting ID 10 to 999) but fails later, the undo-log mechanism must gracefully delete the new PK (999) and restore the old PK (10) without crashing.

In [117]:
print("BEFORE: Member 10 exists:", record_exists(tm, 'sports_db', 'member', 10))
print("BEFORE: Member 999 exists:", record_exists(tm, 'sports_db', 'member', 999))

tx_pk = tm.begin()
try:
    print("\n[Tx] 1. Shifting Member 10's PK to 999...")
    riya = tm.read(tx_pk, 'sports_db', 'member', 10)
    tm.update(tx_pk, 'sports_db', 'member', 10, {**riya, 'MemberID': 999})
    print("Updated: Member 999 exists:", tm.read(tx_pk, 'sports_db', 'member', 999) is not None)
    
    print("[Tx] 2. Triggering intentional failure (invalid sport insert)...")
    # This will fail the CHECK constraint (MaxPlayersPerTeam cannot be negative)
    tm.insert(tx_pk, 'sports_db', 'sport', 99, {
        'SportID': 99, 'SportName': 'Quidditch', 'Category': 'Magic', 'MaxPlayersPerTeam': -5
    }) 
    
    tm.commit(tx_pk)
except ConstraintError as e:
    print(f"\n[ABORT] {e}")

print("\nAFTER ROLLBACK: Member 10 exists:", record_exists(tm, 'sports_db', 'member', 10))
print("AFTER ROLLBACK: Member 999 exists:", record_exists(tm, 'sports_db', 'member', 999))

assert record_exists(tm, 'sports_db', 'member', 10) and not record_exists(tm, 'sports_db', 'member', 999), "PK Rollback Failed!"
print("PK Shift Rollback verified ✓")

[TxManager] BEGIN  tx=e296c307
[TxManager] COMMIT tx=e296c307
BEFORE: Member 10 exists: True
[TxManager] BEGIN  tx=41361d93
[TxManager] COMMIT tx=41361d93
BEFORE: Member 999 exists: False
[TxManager] BEGIN  tx=22c57475

[Tx] 1. Shifting Member 10's PK to 999...
Updated: Member 999 exists: True
[Tx] 2. Triggering intentional failure (invalid sport insert)...
[TxManager] ABORT  tx=22c57475

[ABORT] Insert failed in 'sport': Constraint Error: CHECK failed for 'MaxPlayersPerTeam'='-5'
[TxManager] BEGIN  tx=bfa6a612
[TxManager] COMMIT tx=bfa6a612

AFTER ROLLBACK: Member 10 exists: True
[TxManager] BEGIN  tx=cfa47aa6
[TxManager] COMMIT tx=cfa47aa6
AFTER ROLLBACK: Member 999 exists: False
[TxManager] BEGIN  tx=471da265
[TxManager] COMMIT tx=471da265
[TxManager] BEGIN  tx=90c48237
[TxManager] COMMIT tx=90c48237
PK Shift Rollback verified ✓


---
## C1. Consistency – Constraint Enforcement (CHECK rules)

We attempt to drop `TotalQuantity` of Volleyballs below 0. The transaction manager evaluates the string `CHECK` condition and rejects it.

In [118]:
tx3 = tm.begin()
try:
    print("Attempting to violate CHECK constraint (Quantity < 0)...")
    vb = tm.read(tx3, 'sports_db', 'equipment', 9)
    # Attempting to deduct 50 volleyballs when we only have 10
    tm.update(tx3, 'sports_db', 'equipment', 9, {**vb, 'TotalQuantity': vb['TotalQuantity'] - 50})
    tm.commit(tx3)
except Exception as e:
    print(f"[BLOCKED] {e}\n")

print('Reading records...')
print('Final Volleyball Quantity remains :', read_record(tm, 'sports_db', 'equipment', 9)['TotalQuantity'])

[TxManager] BEGIN  tx=c485256b
Attempting to violate CHECK constraint (Quantity < 0)...
[TxManager] ABORT  tx=c485256b
[BLOCKED] Update failed in 'equipment': Constraint Error: CHECK failed for 'TotalQuantity'='-40'

Reading records...
[TxManager] BEGIN  tx=ba80f910
[TxManager] COMMIT tx=ba80f910
Final Volleyball Quantity remains : 10


---
## C2. Consistency  (Parent-PK Referential Integrity)

We test the reverse Foreign Key checks. If a parent record (like a Sport) is still referenced by child records (like Teams), the engine must block attempts to `DELETE` the parent or `UPDATE` its Primary Key.

In [119]:
# Test A: Trying to Delete a referenced Parent
tx_ghost1 = tm.begin()
try:
    print("Attempting to DELETE Sport 1 (Athletics) which has associated teams...")
    tm.delete(tx_ghost1, 'sports_db', 'sport', 1)
except Exception as e:
    print(f"[BLOCKED DELETE] {e}")

# Test B: Trying to Update a referenced Parent's PK
# We MUST start a new transaction because tx_ghost1 was aborted by the failure!
tx_ghost2 = tm.begin()
try:
    print("\nAttempting to UPDATE Sport 2 (Football) PK to 99...")
    fb = tm.read(tx_ghost2, 'sports_db', 'sport', 2)
    # Trying to change the ID from 2 to 99
    tm.update(tx_ghost2, 'sports_db', 'sport', 2, {**fb, 'SportID': 99})
except Exception as e:
    print(f"[BLOCKED UPDATE] {e}")
    
print("\nVerified: Sport 1 and 2 are safe and cannot be orphaned.")

[TxManager] BEGIN  tx=19d89ff1
Attempting to DELETE Sport 1 (Athletics) which has associated teams...
[TxManager] ABORT  tx=19d89ff1
[BLOCKED DELETE] Foreign Key Violation: Cannot delete '1' from 'sport'. It is still referenced by 'team'.
[TxManager] BEGIN  tx=612d3683

Attempting to UPDATE Sport 2 (Football) PK to 99...
[TxManager] ABORT  tx=612d3683
[BLOCKED UPDATE] Foreign Key Violation: Cannot change PK '2' in 'sport'. It is still referenced by 'team'.

Verified: Sport 1 and 2 are safe and cannot be orphaned.


---
## I1. Isolation – Concurrent Table-Level Modifications

Two threads simultaneously try to grab Badminton Rackets (`EquipmentID` 2). 
Because of the table-level lock manager implemented, they safely queue up sequentially.

In [120]:
results = []

def thread_a():
    tx = tm.begin()
    try:
        racket = tm.read(tx, 'sports_db', 'equipment', 2)
        tm.update(tx, 'sports_db', 'equipment', 2, {**racket, 'TotalQuantity': racket['TotalQuantity'] - 5})
        time.sleep(0.2)   # hold lock briefly to force conflict
        ok, msg = tm.commit(tx)
        results.append(('Thread A', ok, msg))
    except Exception as e: results.append(('Thread A', False, str(e)))

def thread_b():
    time.sleep(0.05)  # start slightly after A to ensure A grabs the lock first
    tx = tm.begin()
    try:
        racket = tm.read(tx, 'sports_db', 'equipment', 2)
        tm.update(tx, 'sports_db', 'equipment', 2, {**racket, 'TotalQuantity': racket['TotalQuantity'] - 3})
        ok, msg = tm.commit(tx)
        results.append(('Thread B', ok, msg))
    except Exception as e:
        tm.rollback(tx)
        results.append(('Thread B', False, str(e)))

t1 = threading.Thread(target=thread_a)
t2 = threading.Thread(target=thread_b)
t1.start(); t2.start()
t1.join(); t2.join()

print('\nThread results:')
for r in results:
    print(f'  {r[0]}: success={r[1]}')

print('\nReading final stock after concurrent updates...')
final_stock = read_record(tm, 'sports_db', 'equipment', 2)['TotalQuantity']
print(f'Racket stock: started=15, expected=7, actual={final_stock}')
assert final_stock == 7, 'Isolation violation!'
print('Isolation verified ✓')

[TxManager] BEGIN  tx=fc067f20
[TxManager] BEGIN  tx=47fd5be3
[TxManager] COMMIT tx=fc067f20
[TxManager] COMMIT tx=47fd5be3

Thread results:
  Thread A: success=True
  Thread B: success=True

Reading final stock after concurrent updates...
[TxManager] BEGIN  tx=c798cee8
[TxManager] COMMIT tx=c798cee8
Racket stock: started=15, expected=7, actual=7
Isolation verified ✓


---
## D1. Durability – Persist committed data across restart

We insert a new `Member`, commit it, and simulate a complete server restart (by loading `database.dat`). The committed member and metadata must survive.

In [121]:
import os

# ── Commit a new Member ───────────────────────────────────────────────
tx_dur = tm.begin()
tm.insert(tx_dur, 'sports_db', 'member', 100, {
    'MemberID': 100, 'Name': 'Test Athlete', 'Image': 'img/test.jpg', 'Age': 21,
    'Email': 'test@iitgn.ac.in', 'ContactNumber': '0000', 'Gender': 'O', 'Role': 'Player', 'JoinDate': '2024-11-01'
})
# COMMIT now automatically calls db.save_to_disk('database.dat')
tm.commit(tx_dur)
print('Member 100 committed.')

# ── Restart the Database ────────────────────────────────────────────────
restored_db = DatabaseManager()

# Initializing the TM triggers load_from_disk() and recover() internally
restored_tm = TransactionManager(restored_db, wal_path='wal.log')

print('\nReading the committed record after restart...')
record = read_record(restored_tm, 'sports_db', 'member', 100)
print(f'Member 100 after restart: {record["Name"]}')
assert record is not None
print('Durability verified ✓')

[TxManager] BEGIN  tx=aff8b135
[TxManager] COMMIT tx=aff8b135
Member 100 committed.

Reading the committed record after restart...
[TxManager] BEGIN  tx=264ed814
[TxManager] COMMIT tx=264ed814
Member 100 after restart: Test Athlete
Durability verified ✓


---
## D2. WAL Recovery – Undo incomplete transaction after crash

We forcefully inject a raw `INSERT` operation directly into the Write-Ahead Log without a corresponding `COMMIT`. When the `TransactionManager` restarts, the recovery logic (incorporating REDO and UNDO phases) will spot the dangling transaction and roll it back.

In [123]:
import json, time as _time

crash_tx_id = 'crash001'
with open('wal_crash.log', 'w') as f:
    f.write(json.dumps({'type': 'BEGIN', 'tx_id': crash_tx_id, 'timestamp': _time.time()}) + '\n')
    f.write(json.dumps({
        'type': 'OP', 'tx_id': crash_tx_id, 'op': 'INSERT', 'db': 'sports_db', 'table': 'equipment',
        'key': 999, 'before': None, 
        'after': {'EquipmentID': 999, 'EquipmentName': 'Ghost Item', 'TotalQuantity': 10, 'EquipmentCondition': 'New', 'SportID': None},
        'timestamp': _time.time()
    }) + '\n')
    # Note: Intentionally missing COMMIT tag as its incomplete due to crash

# Apply the dirty insert to memory so recovery actually has to undo it
eq_tbl.insert({'EquipmentID': 999, 'EquipmentName': 'Ghost Item', 'TotalQuantity': 10, 'EquipmentCondition': 'New', 'SportID': None})
print('Ghost Equipment 999 inserted directly into tree (Simulating crash mid-tx).')
print('Equipment Keys BEFORE recovery:', [k for k, _ in eq_tbl.get_all()])

# Recovery: Booting up a new Transaction Manager tied to the crashed log file
tm_rec = TransactionManager(db, wal_path='wal_crash.log')

print('\nEquipment Keys AFTER recovery:', [k for k, _ in eq_tbl.get_all()])

print('\nReading record 999 after recovery (should be None):', read_record(tm_rec, 'sports_db', 'equipment', 999))
assert not record_exists(tm_rec, 'sports_db', 'equipment', 999), 'Recovery failed - Ghost Equipment still present!'
print('WAL recovery verified ✓')

Ghost Equipment 999 inserted directly into tree (Simulating crash mid-tx).
Equipment Keys BEFORE recovery: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 999]
[Recovery] UNDOING incomplete tx=crash001 (1 ops)
[Recovery] UNDONE 1 incomplete transaction(s)
[Recovery] WAL compacted

Equipment Keys AFTER recovery: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
[TxManager] BEGIN  tx=ba414770
[TxManager] COMMIT tx=ba414770

Reading record 999 after recovery (should be None): None
[TxManager] BEGIN  tx=d663a764
[TxManager] COMMIT tx=d663a764
WAL recovery verified ✓


---
## D3. WAL Recovery – REDO Phase (The Durability Gap)

We simulate a scenario where a transaction `COMMIT`s and writes to the WAL, but the server crashes *before* the engine can save `database.dat`. The recovery process must read the WAL and **REDO** the committed operations that went missing.

In [124]:
import json, time as _time, os

redo_tx_id = 'redo001'

# 1. We artificially write a COMMITTED transaction directly to a new WAL file
with open('wal_redo.log', 'w') as f:
    f.write(json.dumps({'type': 'BEGIN', 'tx_id': redo_tx_id, 'timestamp': _time.time()}) + '\n')
    f.write(json.dumps({
        'type': 'OP', 'tx_id': redo_tx_id, 'op': 'INSERT', 'db': 'sports_db', 'table': 'equipment',
        'key': 888, 'before': None, 
        'after': {'EquipmentID': 888, 'EquipmentName': 'Redo Item', 'TotalQuantity': 5, 'EquipmentCondition': 'New', 'SportID': None},
        'timestamp': _time.time()
    }) + '\n')
    # Note: This one HAS a commit tag, unlike the UNDO test!
    f.write(json.dumps({'type': 'COMMIT', 'tx_id': redo_tx_id, 'timestamp': _time.time()}) + '\n')

# 2. Notice we DO NOT insert the item into the live B+ Tree. 
# It is completely missing from the database right now.
print("Equipment 888 exists BEFORE REDO:", record_exists(tm, 'sports_db', 'equipment', 888))

# 3. We boot up a new Transaction Manager pointed at our redo log.
# It should see the COMMIT, realize the data is missing, and REDO it automatically.
print("\nBooting TransactionManager...")
tm_redo = TransactionManager(db, wal_path='wal_redo.log')

print("\nEquipment 888 exists AFTER REDO:", record_exists(tm_redo, 'sports_db', 'equipment', 888))
assert record_exists(tm_redo, 'sports_db', 'equipment', 888), "REDO Recovery Failed!"
print("WAL REDO recovery verified ✓")

[TxManager] BEGIN  tx=dbf5faea
[TxManager] COMMIT tx=dbf5faea
Equipment 888 exists BEFORE REDO: False

Booting TransactionManager...
[Recovery] REDONE 1 committed transaction(s)
[Recovery] WAL compacted
[TxManager] BEGIN  tx=7907c10f
[TxManager] COMMIT tx=7907c10f

Equipment 888 exists AFTER REDO: True
[TxManager] BEGIN  tx=87213822
[TxManager] COMMIT tx=87213822
WAL REDO recovery verified ✓
